In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os

## Load the data

`X_train`/`X_test` are lists of variable-length sequences (each a `(seq_len, 10)` tensor); `y_train`/`y_test` are the corresponding scalar targets. Make sure `X_train.npy`, `y_train.npy`, `X_test.npy`, and `y_test.npy` (from `homework4_question2_data.zip`) are in the same directory as this notebook before running the cell below.

In [ ]:
dataset_dir = './sequential_data'

# load training and test data
def loadData():
    X_train = np.load(os.path.join(dataset_dir, 'X_train.npy'), allow_pickle=True)
    y_train = np.load(os.path.join(dataset_dir, 'y_train.npy'), allow_pickle=True)
    X_test = np.load(os.path.join(dataset_dir, 'X_test.npy'), allow_pickle=True)
    y_test = np.load(os.path.join(dataset_dir, 'y_test.npy'), allow_pickle=True)

    X_train = [torch.Tensor(x) for x in X_train]  # List of Tensors (SEQ_LEN[i], INPUT_DIM) i=0..NUM_SAMPLES-1
    X_test = [torch.Tensor(x) for x in X_test]  # List of Tensors (SEQ_LEN[i], INPUT_DIM)
    y_train = torch.Tensor(y_train)  # (NUM_SAMPLES, 1)
    y_test = torch.Tensor(y_test)  # (NUM_SAMPLES, 1)

    return X_train, X_test, y_train, y_test


X_train, X_test, y_train, y_test = loadData()
device = y_train.device

## Task 1 (4 points): Implement a Vanilla RNN

Implement a Vanilla RNN architecture by hand (weights are shared across time steps). **You are not allowed to use an RNN layer implementation from any library** (e.g. `nn.RNN`, `nn.RNNCell`) -- only basic layers like `nn.Linear`.

In [ ]:
# Define a Vanilla RNN layer by hand
class RNNLayer(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(RNNLayer, self).__init__()
        self.hidden_size = hidden_size
        self.input_size = input_size
        # BEGIN YOUR CODE HERE (~3 lines)
        # Define the input-to-hidden weights (W_xh), hidden-to-hidden
        # weights (W_hh), and the activation function applied to their
        # combined output (a Vanilla RNN uses tanh).
        self.W_xh = 
        self.W_hh = 
        self.activation = 
        # END YOUR CODE HERE

    def forward(self, x, hidden):
        # BEGIN YOUR CODE HERE (~1 line)
        # Combine the current input x and the previous hidden state,
        # then apply the activation function.
        hidden = 
        # END YOUR CODE HERE
        return hidden


# Define a sequence prediction model using the Vanilla RNN
class SequenceModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SequenceModel, self).__init__()
        self.hidden_size = hidden_size
        # BEGIN YOUR CODE HERE (~2 lines)
        self.rnn = RNNLayer( , )
        self.linear = nn.Linear( , )
        # END YOUR CODE HERE

    def forward(self, input_seq, seq_lengths):
        # BEGIN YOUR CODE HERE (~2 lines)
        # input_seq is a list of (seq_len_b, input_size) tensors, one per
        # sample in the batch (lengths may differ across samples).
        batch_size = 
        last_hidden = torch.zeros( , ).to(device)
        # END YOUR CODE HERE

        for b in range(batch_size):
            hidden = torch.zeros( ).to(device)

            # BEGIN YOUR CODE HERE (~3 lines)
            # Run the RNN layer forward, one timestep at a time, over
            # this sample's full sequence (its own seq_lengths[b] steps),
            # updating `hidden` each step.
            seq_length = 
            for t in range(seq_length):
                hidden = self.rnn(input_seq[ ][ ], hidden)
            # END YOUR CODE HERE

            # Store the last hidden state in the output tensor
            last_hidden[ ] = 

        output = self.linear( )
        return output

## Tasks 2 & 3 (4 points each): Fixed-length variants

`SequenceModelFixedLen` below is shared by both variants -- it is instantiated with `seq_len` set to the dataset's shortest sequence length for **Task 2 (truncated)**, and to the dataset's longest sequence length for **Task 3 (padded)**. Unlike `SequenceModel` above, each timestep gets its **own, non-shared** `RNNLayer` (`self.rnn_layers[t]`), so this model does not tie weights across time.

The constructor parameter `seq_len` is what caps how many timesteps of each sequence actually get processed: in `forward`, each sample only runs for `min(self.seq_len, seq_lengths[b])` steps, i.e. its own length, but never more than `seq_len`. Setting `seq_len` to the shortest training-sequence length (Task 2) therefore truncates every longer sequence down to that length. Setting `seq_len` to the longest sequence length (Task 3) never truncates anything -- every sample simply runs its own true length, which is exactly what "ignoring the padding" amounts to here (a shorter sample never reaches, let alone runs, the extra padded timesteps).

In [ ]:
# Define a sequence prediction model for fixed length sequences, BUT NO SHARED WEIGHTS
class SequenceModelFixedLen(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, seq_len):
        super(SequenceModelFixedLen, self).__init__()
        self.hidden_size = hidden_size
        self.seq_len = seq_len
        # BEGIN YOUR CODE HERE (~2 lines)
        # Note: rnn_layers is wrapped in nn.ModuleList (not a plain
        # Python list) -- nn.Module only auto-registers submodules held
        # in its own attributes/containers like nn.ModuleList, so a bare
        # list here would make every one of these RNNLayers invisible to
        # model.parameters(), and the optimizer would silently never
        # update them.
        self.rnn_layers = nn.ModuleList([RNNLayer( , ) for _ in range( )])
        self.linear = nn.Linear( , )
        # END YOUR CODE HERE

    def forward(self, input_seq, seq_lengths):
        batch_size = 
        last_hidden = torch.zeros(batch_size, self.hidden_size).to(device)

        for b in range(batch_size):
            hidden = torch.zeros( ).to(device)

            # BEGIN YOUR CODE HERE (~3 lines)
            # For Task 2 (truncated), self.seq_len is the shortest
            # sequence length, so every sample runs for exactly
            # self.seq_len steps. For Task 3 (padded), self.seq_len is
            # the longest sequence length; a sample shorter than that
            # should stop at its OWN true length (min(...) below handles
            # both cases -- this is your "ignore the padding" mechanism,
            # since a shorter sample simply never runs its remaining,
            # nonexistent timesteps).
            seq_length = min( , )
            for t in range(seq_length):
                hidden = 
            # END YOUR CODE HERE

            # Store the last hidden state in the output tensor
            last_hidden[ ] = 

        output = self.linear( )
        return output

## Hyperparameters and training loop

In [ ]:
# Define hyperparameters and other settings
input_size = 10  # Replace with the actual dimension of your input features
hidden_size = 64
output_size = 1
num_epochs = 10
learning_rate = 0.001
batch_size = 32

# Sequence lengths across the training set -- needed to pick seq_len for
# the Task 2 (shortest) and Task 3 (longest) SequenceModelFixedLen models.
seq_lengths = [seq.shape[0] for seq in X_train]
min_length = min(seq_lengths)
max_length = max(seq_lengths)

In [ ]:
# Training loop
def train(model, num_epochs, lr, batch_size, X_train, y_train, seq_lengths):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # Task 4 asks you to "monitor and record the training ... loss
    # curves" -- record the average loss per epoch here so you can plot
    # it later (see the plotting cell under Task 4).
    epoch_losses = []

    for epoch in range(num_epochs):
        running_loss = 0.0
        num_batches = 0
        for i in range(0, len(X_train), batch_size):
            inputs = X_train[i:i + batch_size]
            targets = y_train[i:i + batch_size]
            lengths = seq_lengths[i:i + batch_size]

            optimizer.zero_grad()
            outputs = model(inputs, lengths)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            num_batches += 1

        epoch_loss = running_loss / num_batches
        epoch_losses.append(epoch_loss)
        print(f'Epoch {epoch + 1}, Loss: {epoch_loss}')
    return model, epoch_losses

## Task 4 (4 points): Train all three models, then predict before you evaluate

Train the Vanilla RNN, the truncated (shortest-length) model, and the padded (longest-length) model, each using MSE loss for a fixed number of epochs. Monitor and record the training (and validation, if you set aside a validation split) loss curves for each model. `train(...)` returns `(model, epoch_losses)` -- capture both.

Then, *before* looking at test-set performance, predict the **ranking** of the three models by overall test MSE (best to worst). Justify your prediction using the actual mechanics of the three architectures (shared weights vs. fixed-length truncation vs. fixed-length padding) -- not just intuition.

In [ ]:
# BEGIN YOUR CODE HERE
# Initialize and train the Vanilla RNN (SequenceModel).

# END YOUR CODE HERE

In [ ]:
# BEGIN YOUR CODE HERE
# Initialize and train the Sequential NN fixing #timesteps to the
# minimum sequence length (SequenceModelFixedLen with seq_len=min_length).

# END YOUR CODE HERE

In [ ]:
# BEGIN YOUR CODE HERE
# Initialize and train the Sequential NN fixing #timesteps to the
# maximum sequence length (SequenceModelFixedLen with seq_len=max_length).
# NOTE: it is OK to use torch.nn.utils.rnn.pad_sequence; make sure to
# set parameter batch_first correctly.

# END YOUR CODE HERE

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(4, 3))

# BEGIN YOUR CODE HERE
# Plot the recorded per-epoch training (and validation) loss curves for
# all three models on the same axes, for comparison.

# END YOUR CODE HERE

### Task 4 answer

_Your predictions and justification here._

## Task 5 (4 points): Reconcile and analyze

Evaluate all three trained models on the test set and report the overall test MSE for each. Compare the actual ranking to your Task 4 prediction. Where you were right, explain why using the models' mechanics; where you were surprised, propose a concrete explanation. Discuss the trade-offs between shared weights, truncation, and padding for modeling sequences of varying length.

In [ ]:
# BEGIN YOUR CODE HERE
# 1. Evaluate each of the three trained models on the test set (X_test,
#    y_test) using MSE. Compute FRESH sequence lengths from X_test
#    itself (e.g. test_seq_lengths = [seq.shape[0] for seq in X_test])
#    -- don't reuse the training set's seq_lengths variable, since it's
#    still in scope but describes the wrong sequences/order entirely.
# 2. Report the overall test MSE for each model, and the resulting
#    ranking (best to worst).

# END YOUR CODE HERE

### Task 5 answer

_Your analysis here._